### Structured Output(Pydantic, Typedict, Dataclass)
Models provides the response in a format matching the given schema.

In [9]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001A7ABB69690>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A7ABB59650>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [26]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

In [10]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str=Field(description="The title of the movie")
    year: str=Field(description="The year the movie was released")
    director: str=Field(description="The director of the movie")
    rating: str=Field(description="The rating of the movie as per IMDB")


In [11]:
model_with_structure = model.with_structured_output(Movie)
response = model_with_structure.invoke("Provide the details about the movie inception")
response

Movie(title='Inception', year='2010', director='Christopher Nolan', rating='8.8')

### Display Raw and Parsed output

In [15]:
from pydantic import BaseModel, Field


class Movie(BaseModel):
    """A movie with the details."""

    title: str = Field(..., description="The title of the movie")
    year: str = Field(..., description="The year of the movie")
    director: str = Field(..., description="The Director of the movie")
    rating: str = Field(..., description=("The IMDB rating"))

model_with_structure = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke("Provide the details about the movie inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie "Inception". Let me check the tools provided. There\'s a Movie function that requires title, year, director, and rating. I need to find the correct information for each of these parameters.\n\nFirst, the title is obviously "Inception". The year it was released is 2010. The director is Christopher Nolan. As for the IMDB rating, I think it\'s around 8.8. Let me confirm that. Yes, IMDB lists it as 8.8/10. \n\nSo putting that all together, the parameters should be title: "Inception", year: "2010", director: "Christopher Nolan", and rating: "8.8". I need to make sure all required fields are included and formatted correctly as a JSON object within the tool_call tags.\n', 'tool_calls': [{'id': '5w57p8ntn', 'function': {'arguments': '{"director":"Christopher Nolan","rating":"8.8","title":"Inception","year":"2010"}', 'name': 'Movie'}, 'type': 'function'}]}, response_metada

### Nested Structure

In [18]:
from pydantic import BaseModel, Field


class Actor(BaseModel):
    Name: str
    role: str


class MovieDetails(BaseModel):
    title: str
    year: str
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in Millions USD")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Proved the details about the movie inception")
response

MovieDetails(title='Inception', year='2010', cast=[Actor(Name='Leonardo DiCaprio', role='Dom Cobb'), Actor(Name='Joseph Gordon-Levitt', role='Arthur'), Actor(Name='Ellen Page', role='Ariadne'), Actor(Name='Tom Hardy', role='Balthazar')], genres=['Action', 'Science Fiction', 'Thriller'], budget=160.0)

### Typedict

provides simpler alternative using python's builtin typing.

In [24]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A Movie with Details"""
    title: Annotated[str, ..., "Title of the movie"]
    year: Annotated[str, ..., "Year movie was released"]
    director: Annotated[str, ..., "Director of the movie"]
    rating: Annotated[str, ..., "IMDB rating of the movie"]

model_with_structure = model.with_structured_output(MovieDict)
response = model_with_structure.invoke("Provied details about the movie Avengers")
response

{'director': 'Joss Whedon',
 'rating': '8.0',
 'title': 'Avengers',
 'year': '2012'}

In [ ]:
# Nested structrue (can be same as pydantic for nested)


class Actor(TypedDict):
    Name: str
    role: str


class MovieDetails(TypedDict):
    title: str
    year: str
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in Millions USD")


model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Proved the details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'Name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'Name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'Name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'Name': 'Chris Hemsworth', 'role': 'Thor'},
  {'Name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'Name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'genres': ['Action', 'Adventure', 'Science Fiction'],
 'title': 'The Avengers',
 'year': '2012'}

### Integrating with Agents

In [28]:
import os
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [ ]:
# pydantic
from langchain.agents import create_agent
from pydantic import BaseModel, Field


class ContactInfo(BaseModel):
    name: str = Field(description="Name of the person")
    email: str = Field(description="Email of the person")
    phone: str = Field(description="Phone number of the person")


agent = create_agent(
    model="gpt-4o",
    response_format=ContactInfo,
)

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: Dennis, dennis.j@example.com, 8838383838",
            }
        ]
    }
)
print(response)

{'messages': [HumanMessage(content='Extract contact info from: Dennis, dennis.j@example.com, 8838383838', additional_kwargs={}, response_metadata={}, id='3f96560b-7fe7-42c7-9def-99c824eb3f37'), AIMessage(content='{"name":"Dennis","email":"dennis.j@example.com","phone":"8838383838"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 112, 'total_tokens': 137, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_08dbb3f46a', 'id': 'chatcmpl-DJDaztDfjgxXu7teRhr2YhHabzLF5', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ceb38-4ee6-73b3-bbe5-113bded18d41-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 112, 'out

In [32]:
print(response['structured_response'])

name='Dennis' email='dennis.j@example.com' phone='8838383838'


### Data Class  

In [33]:
from dataclasses import dataclass


@dataclass
class ContactInfo:
    name: str = Field(description="Name of the person")
    email: str = Field(description="Email of the person")
    phone: str = Field(description="Phone number of the person")


agent = create_agent(
    model="gpt-4o",
    response_format=ContactInfo,
)

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: Dennis, dennis.j@example.com, 8838383838",
            }
        ]
    }
)
print(response["structured_response"])

ContactInfo(name='Dennis', email='dennis.j@example.com', phone='8838383838')
